In [0]:
# Install all required libraries first
%pip install xgboost shap imbalanced-learn mlflow

In [0]:
%restart_python

In [0]:
# Verify all libraries imported correctly
import mlflow
import xgboost as xgb
import shap
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

print("=" * 50)
print("      LIBRARY VERSIONS")
print("=" * 50)
print(f"  MLflow version:    {mlflow.__version__}")
print(f"  XGBoost version:   {xgb.__version__}")
print(f"  SHAP version:      {shap.__version__}")
print("=" * 50)
print("✅ All libraries ready!")

In [0]:
# Configuration for ML notebook
spark.sql("USE diabetes_medallion")

GOLD_FEATURES  = "diabetes_medallion.gold_features"
GOLD_PREDICTIONS = "diabetes_medallion.gold_predictions"

print("=" * 50)
print("      ML TRAINING CONFIGURATION")
print("=" * 50)
print(f"  Source:        {GOLD_FEATURES}")
print(f"  Predictions:   {GOLD_PREDICTIONS}")
print("=" * 50)
print("✅ Configuration ready!")


In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("=" * 50)
print("      LOADING GOLD FEATURES")
print("=" * 50)

# Step 1 — Load Gold Features table
# Drop non-ML columns
gold_df = spark.table(GOLD_FEATURES) \
    .drop("is_synthetic",
          "gold_timestamp",
          "data_version")

# Step 2 — Convert Spark → Pandas for scikit-learn
print("  Converting Spark → Pandas...")
pdf = gold_df.toPandas()
print(f"  ✅ Shape: {pdf.shape}")

# Step 3 — Separate features and target
X = pdf.drop("Diabetes_binary", axis=1)
y = pdf["Diabetes_binary"]

print(f"\n  Features shape: {X.shape}")
print(f"  Target shape:   {y.shape}")
print(f"  Feature names:  {list(X.columns)}")

# Step 4 — Train / Validation / Test Split
# 70% Train → model learns from this
# 15% Validation → tune and compare models
# 15% Test → final evaluation only

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y     # ← keeps class balance in each split
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("\n" + "=" * 50)
print("      TRAIN/VAL/TEST SPLIT")
print("=" * 50)
print(f"  {'Total rows:':<25} {len(pdf):,}")
print(f"  {'Train rows (70%):':<25} {len(X_train):,}")
print(f"  {'Validation rows (15%):':<25} {len(X_val):,}")
print(f"  {'Test rows (15%):':<25} {len(X_test):,}")

# Verify class balance in each split
print("\n  Class balance per split:")
for name, y_split in [
    ("Train",      y_train),
    ("Validation", y_val),
    ("Test",       y_test)
]:
    pos = y_split.sum()
    neg = (y_split == 0).sum()
    print(f"  {name:12s} → "
          f"0: {neg:,} ({neg/len(y_split)*100:.1f}%) | "
          f"1: {pos:,} ({pos/len(y_split)*100:.1f}%)")

print("=" * 50)
print("✅ Data ready for ML training!")

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

print("=" * 50)
print("      MLFLOW EXPERIMENT SETUP")
print("=" * 50)

# Set experiment name
# All 3 model runs will be grouped here
EXPERIMENT_NAME = "/diabetes_risk_prediction"

mlflow.set_experiment(EXPERIMENT_NAME)

print(f"  ✅ Experiment: {EXPERIMENT_NAME}")
print(f"  ✅ MLflow tracking ready!")
print(f"\n  💡 Find your experiments:")
print(f"     Left sidebar → Experiments")
print(f"     Look for: diabetes_risk_prediction")
print("=" * 50)

In [0]:
# MODEL 1: LOGISTIC REGRESSION
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score
)
import mlflow
import mlflow.sklearn

print("=" * 50)
print("      MODEL 1: LOGISTIC REGRESSION")
print("=" * 50)

# Start MLflow run
with mlflow.start_run(run_name="Logistic_Regression") as run:

    # Step 1 — Define model
    lr_model = LogisticRegression(
        max_iter    = 1000,
        random_state= 42,
        solver      = "lbfgs"
    )

    # Step 2 — Train
    print("  Training Logistic Regression...")
    lr_model.fit(X_train, y_train)
    print("  ✅ Training complete!")

    # Step 3 — Predict on validation set
    val_preds  = lr_model.predict(X_val)
    val_proba  = lr_model.predict_proba(X_val)[:, 1]

    # Step 4 — Calculate metrics
    accuracy  = accuracy_score(y_val, val_preds)
    auc       = roc_auc_score(y_val, val_proba)
    f1        = f1_score(y_val, val_preds)
    precision = precision_score(y_val, val_preds)
    recall    = recall_score(y_val, val_preds)

    # Step 5 — Log parameters to MLflow
    mlflow.log_param("model",       "LogisticRegression")
    mlflow.log_param("max_iter",    1000)
    mlflow.log_param("solver",      "lbfgs")
    mlflow.log_param("train_rows",  len(X_train))
    mlflow.log_param("features",    len(X_train.columns))
    mlflow.log_param("smote",       True)

    # Step 6 — Log metrics to MLflow
    mlflow.log_metric("accuracy",   accuracy)
    mlflow.log_metric("auc",        auc)
    mlflow.log_metric("f1",         f1)
    mlflow.log_metric("precision",  precision)
    mlflow.log_metric("recall",     recall)

    # Step 7 — Log model to MLflow
    signature = infer_signature(X_train, val_preds)
    mlflow.sklearn.log_model(
        lr_model,
        "logistic_regression_model",
        signature = signature
    )

    # Step 8 — Store run ID for comparison later
    lr_run_id = run.info.run_id

    # Print results
    print(f"\n  LOGISTIC REGRESSION RESULTS:")
    print(f"  {'─' * 35}")
    print(f"  {'Accuracy:':<20} {accuracy:.4f}")
    print(f"  {'AUC-ROC:':<20} {auc:.4f}")
    print(f"  {'F1 Score:':<20} {f1:.4f}")
    print(f"  {'Precision:':<20} {precision:.4f}")
    print(f"  {'Recall:':<20} {recall:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_val, val_preds,
          target_names=["No Diabetes", "Diabetes"]))
    print(f"  MLflow Run ID: {lr_run_id}")

print("=" * 50)
print("✅ Logistic Regression complete!")
print("   Logged to MLflow ✅")

In [0]:
# MODEL 2: RANDOM FOREST
from sklearn.ensemble import RandomForestClassifier
import mlflow
import mlflow.sklearn

print("=" * 50)
print("      MODEL 2: RANDOM FOREST")
print("=" * 50)

with mlflow.start_run(run_name="Random_Forest") as run:

    # Step 1 — Define model
    rf_model = RandomForestClassifier(
        n_estimators = 100,
        max_depth    = 10,
        random_state = 42,
        n_jobs       = -1   # use all CPU cores
    )

    # Step 2 — Train
    print("  Training Random Forest...")
    print("  (This may take 1-2 minutes...)")
    rf_model.fit(X_train, y_train)
    print("  ✅ Training complete!")

    # Step 3 — Predict on validation set
    val_preds = rf_model.predict(X_val)
    val_proba = rf_model.predict_proba(X_val)[:, 1]

    # Step 4 — Calculate metrics
    accuracy  = accuracy_score(y_val, val_preds)
    auc       = roc_auc_score(y_val, val_proba)
    f1        = f1_score(y_val, val_preds)
    precision = precision_score(y_val, val_preds)
    recall    = recall_score(y_val, val_preds)

    # Step 5 — Log parameters
    mlflow.log_param("model",        "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth",    10)
    mlflow.log_param("train_rows",   len(X_train))
    mlflow.log_param("features",     len(X_train.columns))
    mlflow.log_param("smote",        True)

    # Step 6 — Log metrics
    mlflow.log_metric("accuracy",  accuracy)
    mlflow.log_metric("auc",       auc)
    mlflow.log_metric("f1",        f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall",    recall)

    # Step 7 — Log model
    signature = infer_signature(X_train, val_preds)
    mlflow.sklearn.log_model(
        rf_model,
        "random_forest_model",
        signature = signature
    )

    # Store run ID
    rf_run_id = run.info.run_id

    # Print results
    print(f"\n  RANDOM FOREST RESULTS:")
    print(f"  {'─' * 35}")
    print(f"  {'Accuracy:':<20} {accuracy:.4f}")
    print(f"  {'AUC-ROC:':<20} {auc:.4f}")
    print(f"  {'F1 Score:':<20} {f1:.4f}")
    print(f"  {'Precision:':<20} {precision:.4f}")
    print(f"  {'Recall:':<20} {recall:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_val, val_preds,
          target_names=["No Diabetes", "Diabetes"]))
    print(f"  MLflow Run ID: {rf_run_id}")

print("=" * 50)
print("✅ Random Forest complete!")
print("   Logged to MLflow ✅")

In [0]:
#MODEL 3: XGBOOST
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn

print("=" * 50)
print("      MODEL 3: XGBOOST")
print("=" * 50)

with mlflow.start_run(run_name="XGBoost") as run:

    # XGBoost optimized for speed + accuracy
    xgb_model = XGBClassifier(
        n_estimators      = 100,
        max_depth         = 6,
        learning_rate     = 0.1,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        random_state      = 42,
        eval_metric       = "logloss",
        verbosity         = 0
    )

    # Train
    print("  Training XGBoost...")
    xgb_model.fit(
        X_train, y_train,
        eval_set        = [(X_val, y_val)],
        verbose         = False
    )
    print("  ✅ Training complete!")

    # Predict
    val_preds = xgb_model.predict(X_val)
    val_proba = xgb_model.predict_proba(X_val)[:, 1]

    # Metrics
    accuracy  = accuracy_score(y_val, val_preds)
    auc       = roc_auc_score(y_val, val_proba)
    f1        = f1_score(y_val, val_preds)
    precision = precision_score(y_val, val_preds)
    recall    = recall_score(y_val, val_preds)

    # Log to MLflow
    mlflow.log_param("model",            "XGBoost")
    mlflow.log_param("n_estimators",     100)
    mlflow.log_param("max_depth",        6)
    mlflow.log_param("learning_rate",    0.1)
    mlflow.log_param("subsample",        0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("train_rows",       len(X_train))
    mlflow.log_param("features",         len(X_train.columns))
    mlflow.log_param("smote",            True)

    mlflow.log_metric("accuracy",  accuracy)
    mlflow.log_metric("auc",       auc)
    mlflow.log_metric("f1",        f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall",    recall)

    signature = infer_signature(X_train, val_preds)
    mlflow.sklearn.log_model(
        xgb_model,
        "xgboost_model",
        signature = signature
    )

    xgb_run_id = run.info.run_id

    # Results
    print(f"\n  XGBOOST RESULTS:")
    print(f"  {'─' * 35}")
    print(f"  {'Accuracy:':<20} {accuracy:.4f}")
    print(f"  {'AUC-ROC:':<20} {auc:.4f}")
    print(f"  {'F1 Score:':<20} {f1:.4f}")
    print(f"  {'Precision:':<20} {precision:.4f}")
    print(f"  {'Recall:':<20} {recall:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_val, val_preds,
          target_names=["No Diabetes", "Diabetes"]))
    print(f"  MLflow Run ID: {xgb_run_id}")

print("=" * 50)
print("✅ XGBoost complete!")
print("   Logged to MLflow ✅")

In [0]:
import pandas as pd
import mlflow

print("=" * 60)
print("      FINAL MODEL COMPARISON")
print("=" * 60)

# Build comparison table
results = {
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [0.7372, 0.7493, 0.7858],
    "AUC-ROC":  [0.8141, 0.8310, 0.8750],
    "F1 Score": [0.7417, 0.7602, 0.7920],
    "Precision":[0.7294, 0.7287, 0.7698],
    "Recall":   [0.7544, 0.7945, 0.8155],
    "MLflow_Run_ID": [lr_run_id, rf_run_id, xgb_run_id]
}

comparison_df = pd.DataFrame(results)

print("\n  Performance Comparison:")
print(comparison_df.to_string(index=False))

# Identify best model
best_model_idx = comparison_df["AUC-ROC"].idxmax()
best_model     = comparison_df.loc[best_model_idx, "Model"]
best_auc       = comparison_df.loc[best_model_idx, "AUC-ROC"]
best_recall    = comparison_df.loc[best_model_idx, "Recall"]

print(f"\n{'=' * 60}")
print(f"  🏆 BEST MODEL: {best_model}")
print(f"  AUC-ROC:  {best_auc:.4f}")
print(f"  Recall:   {best_recall:.4f}")
print(f"  → Catches {best_recall*100:.1f}% of diabetic patients!")
print(f"{'=' * 60}")

# Log comparison to MLflow
with mlflow.start_run(run_name="Model_Comparison"):
    mlflow.log_param("best_model",   best_model)
    mlflow.log_metric("best_auc",    best_auc)
    mlflow.log_metric("best_recall", best_recall)
    mlflow.log_param("lr_run_id",    lr_run_id)
    mlflow.log_param("rf_run_id",    rf_run_id)
    mlflow.log_param("xgb_run_id",   xgb_run_id)

print("✅ Model comparison logged to MLflow!")

In [0]:
import shap
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

print("=" * 50)
print("      SHAP EXPLAINABILITY")
print("=" * 50)

# Use sample of test data for speed
# SHAP on full dataset is slow
SAMPLE_SIZE = 1000
X_sample = X_test.sample(
    n           = SAMPLE_SIZE,
    random_state= 42
)

print(f"  Computing SHAP values on "
      f"{SAMPLE_SIZE} test samples...")

# Create SHAP explainer for XGBoost
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_sample)

print("  ✅ SHAP values computed!")

# Plot 1 — Feature importance bar chart
print("\n  Generating Feature Importance Plot...")
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_sample,
    plot_type = "bar",
    show      = False
)
plt.title(
    "Top Risk Factors for Diabetes Prediction",
    fontsize=14,
    fontweight="bold"
)
plt.tight_layout()
plt.savefig("/tmp/shap_importance.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("  ✅ Feature importance plot saved!")

# Plot 2 — SHAP dot plot
# Shows direction of impact
print("\n  Generating SHAP Dot Plot...")
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_sample,
    show = False
)
plt.title(
    "SHAP Values — Impact on Diabetes Risk",
    fontsize=14,
    fontweight="bold"
)
plt.tight_layout()
plt.savefig("/tmp/shap_dotplot.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("  ✅ SHAP dot plot saved!")

# Print top 5 features by importance
print("\n  TOP 5 RISK FACTORS:")
print("  " + "─" * 40)
import numpy as np
feature_importance = pd.DataFrame({
    "Feature"   : X_sample.columns,
    "Importance": np.abs(shap_values).mean(axis=0)
}).sort_values("Importance", ascending=False)

for i, row in feature_importance.head(5).iterrows():
    print(f"  {row['Feature']:25s} → "
          f"{row['Importance']:.4f}")

print("=" * 50)
print("✅ SHAP explainability complete!")

# Log SHAP plots to MLflow
with mlflow.start_run(run_name="SHAP_Explainability"):
    mlflow.log_artifact("/tmp/shap_importance.png")
    mlflow.log_artifact("/tmp/shap_dotplot.png")
    print("✅ SHAP plots logged to MLflow!")

In [0]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

print("=" * 50)
print("      FINAL TEST SET EVALUATION")
print("      (XGBoost — Best Model)")
print("=" * 50)

# IMPORTANT:
# This is the ONLY time we touch test set!
# Sealed envelope now opened! 🔒→🔓

# Predict on test set
test_preds = xgb_model.predict(X_test)
test_proba = xgb_model.predict_proba(X_test)[:, 1]

# Final metrics
test_accuracy  = accuracy_score(y_test, test_preds)
test_auc       = roc_auc_score(y_test, test_proba)
test_f1        = f1_score(y_test, test_preds)
test_precision = precision_score(y_test, test_preds)
test_recall    = recall_score(y_test, test_preds)

print(f"\n  FINAL TEST RESULTS:")
print(f"  {'─' * 35}")
print(f"  {'Accuracy:':<20}  {test_accuracy:.4f}")
print(f"  {'AUC-ROC:':<20}  {test_auc:.4f}")
print(f"  {'F1 Score:':<20}  {test_f1:.4f}")
print(f"  {'Precision:':<20}  {test_precision:.4f}")
print(f"  {'Recall:':<20}  {test_recall:.4f}")

# Compare Val vs Test
print(f"\n  VALIDATION vs TEST COMPARISON:")
print(f"  {'─' * 40}")
print(f"  {'Metric':<15} {'Validation':>12} "
      f"{'Test':>12} {'Difference':>12}")
print(f"  {'─' * 40}")

metrics = {
    "Accuracy":  (0.7858, test_accuracy),
    "AUC-ROC":   (0.8750, test_auc),
    "F1 Score":  (0.7920, test_f1),
    "Precision": (0.7698, test_precision),
    "Recall":    (0.8155, test_recall)
}

for metric, (val_score, test_score) in metrics.items():
    diff = test_score - val_score
    flag = "✅" if abs(diff) < 0.02 else "⚠️"
    print(f"  {metric:<15} {val_score:>12.4f} "
          f"{test_score:>12.4f} "
          f"{diff:>+11.4f} {flag}")

# Confusion Matrix
print(f"\n  Confusion Matrix:")
cm = confusion_matrix(y_test, test_preds)
print(f"  {'─' * 35}")
print(f"  True Negatives:  {cm[0][0]:,} "
      f"(correctly said NO diabetes)")
print(f"  False Positives: {cm[0][1]:,} "
      f"(said diabetes, actually NO)")
print(f"  False Negatives: {cm[1][0]:,} "
      f"(missed diabetes cases!) ⚠️")
print(f"  True Positives:  {cm[1][1]:,} "
      f"(correctly caught diabetes!)")

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot      = True,
    fmt        = "d",
    cmap       = "Blues",
    xticklabels= ["No Diabetes", "Diabetes"],
    yticklabels= ["No Diabetes", "Diabetes"],
    ax         = ax
)
ax.set_title(
    "XGBoost Confusion Matrix — Test Set",
    fontsize=14,
    fontweight="bold"
)
ax.set_ylabel("Actual", fontsize=12)
ax.set_xlabel("Predicted", fontsize=12)
plt.tight_layout()
plt.savefig("/tmp/confusion_matrix.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("\n  ✅ Confusion matrix saved!")

# Log final results to MLflow
with mlflow.start_run(run_name="XGBoost_Final_Test"):
    mlflow.log_param("model",      "XGBoost")
    mlflow.log_param("dataset",    "test_set")
    mlflow.log_metric("accuracy",  test_accuracy)
    mlflow.log_metric("auc",       test_auc)
    mlflow.log_metric("f1",        test_f1)
    mlflow.log_metric("precision", test_precision)
    mlflow.log_metric("recall",    test_recall)
    mlflow.log_artifact("/tmp/confusion_matrix.png")
    print("✅ Final results logged to MLflow!")

print("=" * 50)
print("✅ Final evaluation complete!")

In [0]:
from pyspark.sql.functions import (
    lit, current_timestamp, col,
    sum as spark_sum, count,
    round as spark_round
)
import pandas as pd

print("=" * 50)
print("      SAVING PREDICTIONS TO GOLD")
print("=" * 50)

# Step 1 — Build predictions dataframe
predictions_pdf = X_test.copy()
predictions_pdf["actual_label"]         = y_test.values
predictions_pdf["predicted_label"]      = test_preds
predictions_pdf["diabetes_probability"] = test_proba

# Step 2 — Add risk tier
predictions_pdf["risk_tier"] = pd.cut(
    predictions_pdf["diabetes_probability"],
    bins   = [0, 0.30, 0.60, 1.0],
    labels = ["Low Risk", "Medium Risk", "High Risk"]
)

# Step 3 — Add correctness flag
predictions_pdf["prediction_correct"] = (
    predictions_pdf["actual_label"] ==
    predictions_pdf["predicted_label"]
)

# Step 4 — Convert to Spark
predictions_df = spark.createDataFrame(
    predictions_pdf
) \
.withColumn("model_name",    lit("XGBoost"))        \
.withColumn("model_version", lit("1.0"))             \
.withColumn("prediction_ts", current_timestamp())

# Step 5 — Save to Gold layer
predictions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_PREDICTIONS)

# Step 6 — Verify
verify_df = spark.table(GOLD_PREDICTIONS)
print(f"  ✅ Predictions saved to Gold!")
print(f"  Table:   {GOLD_PREDICTIONS}")
print(f"  Rows:    {verify_df.count():,}")
print(f"  Columns: {len(verify_df.columns)}")

# Step 7 — Risk tier distribution
print(f"\n  Risk Tier Distribution:")
verify_df.groupBy("risk_tier") \
         .count() \
         .orderBy("risk_tier") \
         .show()

# Step 8 — Accuracy by risk tier
print(f"\n  Accuracy by Risk Tier:")
verify_df.groupBy("risk_tier") \
    .agg(
        spark_sum(
            col("prediction_correct").cast("int")
        ).alias("correct"),
        count("*").alias("total")
    ) \
    .withColumn(
        "accuracy_pct",
        spark_round(
            col("correct") * 100.0 / col("total"), 2
        )
    ) \
    .orderBy("risk_tier") \
    .show()

print("=" * 50)
print("✅ Predictions stored in Gold layer!")
print("   Pipeline ↔ AI loop CLOSED! 🎯")